# 200x160 Run Comparison Diagnostics

Standalone notebook for balanced comparison figures:
1. Joint frequency `qv` vs `qc`
2. Diameter-resolved histogram summary (`qfw`/`qw`)
3. Threshold occurrence/exceedance (`qi+qs`, `qc`) with run-difference panels

## Compute Playbook for Very Large NetCDF (1.6 TB class)

Use these rules by default for diagnostics notebooks:

- Avoid full `persist()` on high-dimensional datasets (`time,z,y,x[,bin]`).
- Avoid `.values` on large lazy arrays unless aggressively downsampled first.
- Convert heavy science fields to `float32` for diagnostics (`.astype('float32')`).
- Derive histogram bins from sampled subsets (stride in `time/altitude/lat/lon/diameter`).
- Compute expensive histograms on sampled data, not full-resolution 5D volumes.
- Keep `.compute()` targeted to compact intermediates (histograms, 2D/3D summaries).

Why this matters:

- A single `time,z,y,x,bin` variable at your dimensions is roughly **300+ GiB** in `float64`.
- Two runs + temporary arrays can exceed node RAM and crash the kernel.
- Stride-based summaries are usually stable enough for figure diagnostics at a fraction of the cost.

Recommended execution order:

1. Restart kernel before heavy runs.
2. Run cells sequentially (do not launch overlapping heavy cells).
3. If runtime/memory is high, increase stride before requesting more resources.
4. Only move to larger allocations when full-resolution science output is required.

In [1]:
import os
import glob
import json
from time import perf_counter

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from xhistogram.xarray import histogram as xhist
from tqdm.auto import tqdm

from dask.diagnostics.progress import ProgressBar

from utilities.model_helpers import fetch_3d_data, convert_units_3d
from utilities.namelist_metadata import update_dataset_metadata
from utilities import define_bin_boundaries

xr.set_options(keep_attrs=True)

model_data_root = '/work/bb1262/user/schimmel/cosmo-specs-torch/cosmo-specs-runs/'
cs_runs = [
    ['cs-eriswil__20260123_180947', '200x160', 0],
    ['cs-eriswil__20260123_180947', '200x160', 1],
]
tracking_by_var = 'qi+qs'

In [2]:
def _load_run_mod(cs_run, ep_dom, cs_run_idx, tracking_by_var='qi+qs'):
    model_data_path = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/ensemble_output/{cs_run}/'
    extpar_file = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/COS_in/extPar_Eriswil_{ep_dom}.nc'
    json_file = glob.glob(model_data_path + '*.json')[0]

    with open(json_file, 'r') as f:
        meta = json.load(f)

    flist_3d = sorted(glob.glob(model_data_path + '3D_??????????????.nc'))
    exp_names = [f.split('/')[-1].split('_')[-1].split('.')[0] for f in flist_3d]
    flare_exp_name = [exp for exp in exp_names if meta[exp]['INPUT_ORG']['sbm_par']['lflare']][cs_run_idx]
    flare_exp_nc_file = [f for f in flist_3d if flare_exp_name in f][0]

    reduced_domain = dict(
        latitude=slice(None, 47.07425 + 2.0 * 0.001),
        longitude=slice(None, 7.90522 + 2.0 * 0.001),
    )

    ds_3d = fetch_3d_data(
        flare_exp_nc_file,
        extpar_file,
        meta[flare_exp_name]['INPUT_ORG'],
        var_sets=['meteo', 'spec'],
        chunks={'time': 1, 'altitude': 1},
    )
    ds_3d = update_dataset_metadata(ds_3d)
    ds_3d = ds_3d.isel(altitude=slice(80, None)).sel(reduced_domain)
    ds_3d = convert_units_3d(ds_3d, ds_3d['rho'])

    mod = ds_3d[['qv', 'qc', 'qi', 'qs', 'qfw', 'qw', 'nf', 'nw']].astype('float32')
    d_um = define_bin_boundaries() * 1.0e6 * 2.0
    mod = mod.assign_coords({'diameter_edges': xr.DataArray(d_um, dims='diameter_edges')})
    mod[tracking_by_var] = xr.where(mod['qi'] + mod['qs'] > 0, mod['qi'] + mod['qs'], np.nan)
    mod[tracking_by_var].attrs['units'] = mod['qi'].attrs.get('units', '')
    return mod, flare_exp_name


def _sample_for_bins(da, stride=None):
    stride = stride or {'time': 16, 'altitude': 4, 'latitude': 8, 'longitude': 8, 'diameter': 4}
    sel = {dim: slice(None, None, stride[dim]) for dim in da.dims if dim in stride}
    return da.isel(sel)


def _safe_log_bins(da, nbins=72, stride=None):
    sample = _sample_for_bins(da, stride=stride)
    values = np.asarray(sample.where(np.isfinite(sample) & (sample > 0)).compute().values).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.logspace(-12, -6, nbins)
    vmin, vmax = np.nanpercentile(values, [0.5, 99.5])
    vmin = max(float(vmin), np.finfo(float).tiny)
    vmax = max(float(vmax), vmin * 10.0)
    return np.logspace(np.log10(vmin), np.log10(vmax), nbins)


def _compute_joint_freq(mod, xbins, ybins):
    h = xhist(mod['qv'], mod['qc'], bins=[xbins, ybins]).compute()
    return h / h.sum()


def _compute_diameter_resolved_qfreq(mod, xbins, ybins, stride=None, show_progress=True):
    stride = stride or {'time': 2, 'altitude': 2, 'latitude': 4, 'longitude': 4, 'diameter': 2}
    sampled = mod.isel(
        time=slice(None, None, stride['time']),
        altitude=slice(None, None, stride['altitude']),
        latitude=slice(None, None, stride['latitude']),
        longitude=slice(None, None, stride['longitude']),
        diameter=slice(None, None, stride['diameter']),
    )
    rows = []
    iterator = tqdm(range(sampled.diameter.size), desc='diameter bins', leave=True) if show_progress else range(sampled.diameter.size)
    for ibin in iterator:
        mo = sampled.isel(diameter=ibin)
        h = xhist(mo['qfw'], mo['qw'], bins=[xbins, ybins]).sum('qw_bin').compute().expand_dims('diameter')
        rows.append(h)
    ds = xr.concat(rows, dim='diameter').assign_coords({'diameter': sampled.diameter})
    return ds / ds.sum('qfw_bin')


def _compute_exceedance(mod, var, threshold):
    da = xr.where(mod[var] >= threshold, 1.0, 0.0).mean(dim=('latitude', 'longitude'))
    return da.compute()

In [ ]:
run_cfg = {
    'nbins': 64,
    'bin_stride': {'time': 16, 'altitude': 4, 'latitude': 8, 'longitude': 8, 'diameter': 4},
    'diam_stride': {'time': 2, 'altitude': 2, 'latitude': 4, 'longitude': 4, 'diameter': 2},
}

start_all = perf_counter()

def _log_stage(label, t0):
    dt = perf_counter() - t0
    print(f'[stage] {label:<30} {dt:7.1f} s')
    return perf_counter()

print('[info] Starting comparison workflow...')
t0 = perf_counter()
run_data = []
for cs_run, ep_dom, cs_run_idx in tqdm(cs_runs, desc='load runs'):
    mod_i, flare_name_i = _load_run_mod(cs_run, ep_dom, cs_run_idx, tracking_by_var=tracking_by_var)
    run_data.append({'run_id': cs_run, 'flare_exp_name': flare_name_i, 'mod': mod_i})
t0 = _log_stage('load runs', t0)

mod1, mod2 = run_data[0]['mod'], run_data[1]['mod']
label1, label2 = run_data[0]['run_id'], run_data[1]['run_id']

qv_bins = _safe_log_bins(xr.concat([mod1['qv'], mod2['qv']], dim='stack_qv'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'])
qc_bins = _safe_log_bins(xr.concat([mod1['qc'], mod2['qc']], dim='stack_qc'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'])
qfw_bins = _safe_log_bins(xr.concat([mod1['qfw'], mod2['qfw']], dim='stack_qfw'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'])
qw_bins = _safe_log_bins(xr.concat([mod1['qw'], mod2['qw']], dim='stack_qw'), nbins=run_cfg['nbins'], stride=run_cfg['bin_stride'])
t0 = _log_stage('derive bins', t0)

joint1 = _compute_joint_freq(mod1, qv_bins, qc_bins)
joint2 = _compute_joint_freq(mod2, qv_bins, qc_bins)
joint_diff = joint2 - joint1
t0 = _log_stage('joint histogram compute', t0)

# Figure A: Joint qv/qc frequency
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
vmax_joint = float(np.nanmax([joint1.max().values, joint2.max().values]))
vmin_joint = max(vmax_joint * 1e-5, np.finfo(float).tiny)

for i, (da, ttl) in enumerate([(joint1, label1), (joint2, label2)]):
    xr.where(da > 0, da, np.nan).plot(
        ax=ax[i],
        xscale='log',
        yscale='log',
        cmap='viridis',
        norm=LogNorm(vmin=vmin_joint, vmax=vmax_joint),
        cbar_kwargs={'label': 'joint frequency'},
    )
    ax[i].set_title(f'Joint freq qv/qc: {ttl}')
    ax[i].set_xlabel('qv')
    ax[i].set_ylabel('qc')

vd = float(np.nanmax(np.abs(joint_diff.values)))
if vd == 0:
    vd = 1e-12
joint_diff.plot(
    ax=ax[2],
    xscale='log',
    yscale='log',
    cmap='coolwarm',
    vmin=-vd,
    vmax=vd,
    cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
)
ax[2].set_title('Joint freq difference')
ax[2].set_xlabel('qv')
ax[2].set_ylabel('qc')
fig.savefig('fig-comparison-a-joint-freq-200x160.png', dpi=300)
t0 = _log_stage('figure A saved', t0)

# Figure B: Diameter-resolved qfw/qw summary
print('[info] Computing diameter-resolved histograms...')
diam_q1 = _compute_diameter_resolved_qfreq(mod1, qfw_bins, qw_bins, stride=run_cfg['diam_stride'], show_progress=True)
diam_q2 = _compute_diameter_resolved_qfreq(mod2, qfw_bins, qw_bins, stride=run_cfg['diam_stride'], show_progress=True)
diam_qdiff = diam_q2 - diam_q1
t0 = _log_stage('diameter histogram compute', t0)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
vmax_d = float(np.nanmax([diam_q1.max().values, diam_q2.max().values]))
vmin_d = max(vmax_d * 1e-5, np.finfo(float).tiny)

for i, (da, ttl) in enumerate([(diam_q1, label1), (diam_q2, label2)]):
    xr.where(da > 0, da, np.nan).plot(
        ax=ax[i],
        x='diameter',
        y='qfw_bin',
        xscale='log',
        yscale='log',
        cmap='magma',
        norm=LogNorm(vmin=vmin_d, vmax=vmax_d),
        cbar_kwargs={'label': 'freq(qfw | diameter)'},
    )
    ax[i].set_title(f'Diameter-resolved qfw/qw: {ttl}')
    ax[i].set_xlabel('diameter (um)')
    ax[i].set_ylabel('qfw')

vdd = float(np.nanmax(np.abs(diam_qdiff.values)))
if vdd == 0:
    vdd = 1e-12
diam_qdiff.plot(
    ax=ax[2],
    x='diameter',
    y='qfw_bin',
    xscale='log',
    yscale='log',
    cmap='coolwarm',
    vmin=-vdd,
    vmax=vdd,
    cbar_kwargs={'label': f'freq diff ({label2} - {label1})'},
)
ax[2].set_title('Diameter-resolved difference')
ax[2].set_xlabel('diameter (um)')
ax[2].set_ylabel('qfw')
fig.savefig('fig-comparison-b-diameter-resolved-hist-200x160.png', dpi=300)
t0 = _log_stage('figure B saved', t0)

# Figure C: Threshold occurrence time-height
thresholds = {'qi+qs': 1e-8, 'qc': 1e-8}
occ = {}
for var, th in tqdm(thresholds.items(), desc='threshold occurrence'):
    occ[(label1, var)] = _compute_exceedance(mod1, var, th)
    occ[(label2, var)] = _compute_exceedance(mod2, var, th)
t0 = _log_stage('threshold occurrence compute', t0)

fig, ax = plt.subplots(2, 3, figsize=(15, 7.2), constrained_layout=True, sharex=True, sharey=True)
for r, var in enumerate(['qi+qs', 'qc']):
    da1 = occ[(label1, var)]
    da2 = occ[(label2, var)]
    dad = da2 - da1

    for c, (da, ttl, cmap) in enumerate([
        (da1, f'{label1}: {var} >= {thresholds[var]:.1e}', 'viridis'),
        (da2, f'{label2}: {var} >= {thresholds[var]:.1e}', 'viridis'),
        (dad, f'diff ({label2} - {label1})', 'coolwarm'),
    ]):
        kw = dict(x='time', y='altitude', cmap=cmap, add_colorbar=True)
        if c < 2:
            kw['vmin'] = 0.0
            kw['vmax'] = 1.0
            kw['cbar_kwargs'] = {'label': 'occurrence frequency'}
        else:
            vm = float(np.nanmax(np.abs(dad.values)))
            if vm == 0:
                vm = 1e-12
            kw['vmin'] = -vm
            kw['vmax'] = vm
            kw['cbar_kwargs'] = {'label': 'frequency difference'}
        da.plot(ax=ax[r, c], **kw)
        ax[r, c].set_title(ttl)
        ax[r, c].set_xlabel('time')
        ax[r, c].set_ylabel('altitude')

fig.savefig('fig-comparison-c-threshold-occurrence-200x160.png', dpi=300)
t0 = _log_stage('figure C saved', t0)

print('Saved: fig-comparison-a-joint-freq-200x160.png')
print('Saved: fig-comparison-b-diameter-resolved-hist-200x160.png')
print('Saved: fig-comparison-c-threshold-occurrence-200x160.png')
print(f'[done] total elapsed {perf_counter() - start_all:7.1f} s')

[info] Starting comparison workflow...


load runs:   0%|          | 0/2 [00:00<?, ?it/s]

[stage] load runs                          2.0 s
[stage] derive bins                       34.0 s
[stage] joint histogram compute          113.9 s
[stage] figure A saved                     2.4 s
[info] Computing diameter-resolved histograms...


diameter bins:   0%|          | 0/33 [00:00<?, ?it/s]